# Thailand HV Transmission Grid from OpenStreetMap

This notebook demonstrates how to:
1. Fetch HV transmission grid data from OSM
2. Visualize the grid topology
3. Convert to pandapower format
4. Integrate with GridTokenX simulator

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import json
import sys
from pathlib import Path

# Add backend scripts to path
sys.path.append('../scripts')

# Configure OSMnx
ox.settings.use_cache = True
ox.settings.log_console = False

print("Dependencies loaded successfully")

## Step 1: Fetch HV Transmission Lines from OSM

In [ ]:
# Thailand HV voltage levels (in volts)
HV_VOLTAGES = ["500000", "230000", "115000"]

# OSM tags for HV transmission
hv_tags = {
    "power": "line",
    "voltage": HV_VOLTAGES,
}

# Bangkok coordinates
bangkok_lat = 13.757559
bangkok_lon = 100.688337

# Fetch data (100km radius)
print(f"Fetching HV transmission lines around Bangkok...")
grid_gdf = ox.features_from_point(
    (bangkok_lat, bangkok_lon),
    tags=hv_tags,
    dist=100000
)

print(f"Found {len(grid_gdf)} HV transmission lines")

## Step 2: Explore the Data

In [ ]:
# Check available columns (OSM tags)
print("Available OSM tags:")
for col in sorted(grid_gdf.columns):
    if col != 'geometry':
        non_null = grid_gdf[col].notna().sum()
        print(f"  {col}: {non_null}/{len(grid_gdf)} values")

In [ ]:
# Voltage distribution
if 'voltage' in grid_gdf.columns:
    print("\nVoltage level distribution:")
    voltage_counts = grid_gdf['voltage'].value_counts()
    for volt, count in voltage_counts.head(10).items():
        kv = int(volt) / 1000 if str(volt).isdigit() else volt
        print(f"  {kv:>6} kV: {count} lines")

In [ ]:
# Operator distribution
if 'operator' in grid_gdf.columns:
    print("\nOperator distribution:")
    operator_counts = grid_gdf['operator'].value_counts()
    for op, count in operator_counts.head(10).items():
        print(f"  {op}: {count}")

## Step 3: Visualize the Grid

In [ ]:
# Plot HV transmission lines
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Color by voltage level
if 'voltage' in grid_gdf.columns:
    grid_gdf['voltage_kv'] = pd.to_numeric(grid_gdf['voltage'], errors='coerce') / 1000
    
    # Group by voltage for coloring
    for voltage_kv, group in grid_gdf.groupby('voltage_kv'):
        if pd.notna(voltage_kv):
            group.plot(ax=ax, label=f"{int(voltage_kv)} kV", linewidth=2)
else:
    grid_gdf.plot(ax=ax, linewidth=2)

ax.set_title("Thailand HV Transmission Grid (from OpenStreetMap)", fontsize=14, fontweight='bold')
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4: Save to GeoJSON

In [ ]:
# Save to GeoJSON
output_file = Path('../data/thailand_hv_grid.geojson')
output_file.parent.mkdir(parents=True, exist_ok=True)

grid_gdf.to_file(output_file, driver='GeoJSON')
print(f"Saved to {output_file}")

## Step 5: Convert to Pandapower Format

In [ ]:
# Import conversion functions
from fetch_thailand_hv_grid import convert_to_pandapower_format

# Convert GeoJSON to pandapower format
pandapower_data = convert_to_pandapower_format(
    str(output_file),
    '../data/pandapower_hv_grid.json'
)

print(f"\nConversion complete:")
print(f"  Buses: {pandapower_data['metadata']['total_buses']}")
print(f"  Lines: {pandapower_data['metadata']['total_lines']}")
print(f"  Voltage levels: {pandapower_data['metadata']['voltage_levels_kv']}")

## Step 6: Create Pandapower Network

In [ ]:
import pandapower as pp

# Load converted data
with open('../data/pandapower_hv_grid.json') as f:
    osm_grid = json.load(f)

# Create empty pandapower network
net = pp.create_empty_network()

# Create buses
bus_id_to_idx = {}
for bus in osm_grid['buses']:
    bus_idx = pp.create_bus(
        net,
        vn_kv=bus['voltage_kv'],
        name=bus['id'],
        type='b',
    )
    bus_id_to_idx[bus['id']] = bus_idx

print(f"Created {len(net.bus)} buses")

In [ ]:
# Create lines
for line in osm_grid['lines']:
    from_bus_idx = bus_id_to_idx[line['from_bus']]
    to_bus_idx = bus_id_to_idx[line['to_bus']]
    
    pp.create_line(
        net,
        from_bus=from_bus_idx,
        to_bus=to_bus_idx,
        length_km=line['length_km'],
        std_type='NAYY 4x120 SE',
        r_ohm_per_km=line['r_ohm_per_km'],
        x_ohm_per_km=line['x_ohm_per_km'],
        name=f"line_{line.get('osm_id', 'unknown')}",
    )

print(f"Created {len(net.line)} lines")
print(f"\nNetwork summary:")
print(net)

## Step 7: Run Power Flow Analysis

In [ ]:
# Add external grid connection (slack bus)
if len(net.bus) > 0:
    slack_bus_idx = net.bus.index[0]
    
    pp.create_ext_grid(
        net,
        bus=slack_bus_idx,
        vm_pu=1.0,
        va_degree=0.0,
    )
    
    print(f"Added external grid at bus {slack_bus_idx}")

# Run power flow
try:
    pp.runpp(net)
    print("\nPower flow converged!")
    print(f"Slack bus results:")
    print(f"  Voltage: {net.res_bus.vm_pu.iloc[0]:.4f} pu")
    print(f"  Angle: {net.res_bus.va_degree.iloc[0]:.2f}°")
except Exception as e:
    print(f"Power flow failed: {e}")

## Step 8: Integrate with GridTokenX Simulator

In [ ]:
# Example: Use OSM-derived grid in simulator
# from smart_meter_simulator.adapters.pandapower_adapter import PandapowerAdapter
# from smart_meter_simulator.core.engine import SimulationEngine

# adapter = PandapowerAdapter(net=net)
# engine = SimulationEngine(pandapower_adapter=adapter)

# print("Simulator initialized with OSM grid")

## Next Steps

1. **Fetch larger area:** Use bounding box to cover entire Thailand
2. **Add substations:** Query `power=substation` separately
3. **Validate against EGAT data:** Compare with official grid maps
4. **Contribute to OSM:** Map missing transmission lines
5. **Run state estimation:** Use simulator's WLS/Iwamoto algorithms
6. **P2P trading simulation:** Test market dynamics on realistic grid

See documentation: `docs/guides/using_osm_data_thailand_grid.md`